In [1]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import os 
import json

# <span style="color:#AE74D4"> Using Shimming Toolbox for realtime f0xyz shimming! </span> </br>
This notebook assumes you can connect to your scanner and get the dicom files directly after the sequence is ran. <br>
The steps are: </br>
- 1. You will ran a fieldmapping sequence that will generate xx_gre_mag and xx_gre_phs files 
- 2. Once it's finished running, we need to sort the dicoms.
- 3. Then we need to go from dcm to nifti
- 4. Once we've done this, we need to generate a mask of our region of interest (ROI) 
- 5. With all the pre-requisites ready, we can calculate a total fieldmap 
- 6. Calculate the new shimm coefficients with Shimming Toolbox, update the ICE builder with the new custom coefficients and run your desired gre sequence with custom shimming!

## <span style="color:#AE74D4"> Locate shared dicom folder SYNGO TRANSFER </span> </br>

In [5]:
# Locate the transfer directory 
transfer_home_directory = r"/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER"
# Then, after the sequence is ran, a file will be created e.g. 20260525.phantom.2026.05.25_12_34_40_DST_1.3.12.2.1107.5.99.3
# This will have the dcm files for anything you've ran while the transfer is ON (so localizers, T1w or other)

In [ ]:
/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/20260528.chi_012.2026.05.28_15_05_28_DST_1.3.12.2.1107.5.99.3

In [6]:
# Complete this once the sequences you need have ran

experiment_fn = "20260528.chi_012.2026.05.28_15_05_28_DST_1.3.12.2.1107.5.99.3" 

In [7]:
print(os.path.exists(os.path.join(transfer_home_directory,experiment_fn)))

True


In [14]:
custom_directory = r"/Users/mclogar/msc_data/SYNGO_TRANSFER/phantom_may_25th/20260525.phantom.2026.05.25_12_34_40_DST_1.3.12.2.1107.5.99.3"

# In case you want to use a custom directory or you've already moved the data

In [11]:
%cd {transfer_home_directory}
%cd {experiment_fn}
!ls

/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER
/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/20260528.chi_012.2026.05.28_15_05_28_DST_1.3.12.2.1107.5.99.3
001_000002_000001_1.3.12.2.1107.5.2.43.167006.2026052815370760030500453.dcm
001_000003_000001_1.3.12.2.1107.5.2.43.167006.2026052815432349417402050.dcm
001_000004_000001_1.3.12.2.1107.5.2.43.167006.2026052815550951980802723.dcm
001_000004_000002_1.3.12.2.1107.5.2.43.167006.2026052815550953169902727.dcm
001_000005_000001_1.3.12.2.1107.5.2.43.167006.2026052815550952601402725.dcm
001_000005_000002_1.3.12.2.1107.5.2.43.167006.2026052815550953366502728.dcm
001_000006_000001_1.3.12.2.1107.5.2.43.167006.2026052815595597217603165.dcm
001_000006_000002_1.3.12.2.1107.5.2.43.167006.2026052815595598465503169.dcm
001_000006_000003_1.3.12.2.1107.5.2.43.167006.2026052815595598955103171.dcm
001_000006_000004_1.3.12.2.1107.5.2.43.167006.2026052815595599455703173.dcm
001_000006_000005_1.3.12.2.1107.5.2.43.167006.202605281559559997150317

## <span style="color:#AE74D4"> I. Sort dicoms </span> </br>

In [23]:
scan_directory = os.path.join(transfer_home_directory, experiment_fn)
print(scan_directory)
print(os.path.exists(scan_directory))

/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/20260528.chi_012.2026.05.28_15_05_28_DST_1.3.12.2.1107.5.99.3
True


In [24]:
outpath_for_sorted_dicoms = os.path.join(transfer_home_directory, "I_sorted_dicoms_may29th")

In [25]:
# I. Sort dicoms
!st_sort_dicoms -i {scan_directory} -o {outpath_for_sorted_dicoms}

INFO:shimmingtoolbox.cli.sort_dicoms:Successfully sorted the DICOMs


## <span style="color:#AE74D4"> II. Dicom to nifti </span> </br>

In [26]:
outpath_for_niftis = os.path.join(transfer_home_directory, "I_niftis_may_28th")
subject_tag = "chi-012"

In [27]:
print(outpath_for_niftis)

/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th


In [28]:
# Create niftis for all dicom files on the sorted dicom folder
!st_dicom_to_nifti -i {outpath_for_sorted_dicoms} -o {outpath_for_niftis} --subject chi12

INFO:shimmingtoolbox.dicom_to_nifti:dcm2bids version: 3.2.0
INFO:dcm2bids.utils.utils:Running: dcm2niix -b y -ba y -z y -f %3s_%f_%p_%t -o /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/tmp_dcm2bids/sub-chi12 /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_sorted_dicoms_may29th
INFO:dcm2bids.dcm2niix_gen:Log from dcm2niix execution
Chris Rorden's dcm2niiX version v1.0.20250505  Clang15.0.0 x86-64 (64-bit MacOS)
Found 34 DICOM file(s)
Slices not stacked: echo varies (TE 15, 35; echo 3, 7). Use 'merge 2D slices' option to force stacking
Convert 1 DICOM as /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/tmp_dcm2bids/sub-chi12/008_I_sorted_dicoms_may29th_2D_7meGRE_no_custom_shim_20260528150928_e3 (384x384x16x1)
Convert 1 DICOM as /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/tmp_dcm2bids/sub-chi12/008_I_sorted_dicoms_may29th_2D_7meGRE_no_custom_shim_20260528150928_e7 (384x384x16x1)
Convert 1 DICOM as /Users/

## <span style="color:#AE74D4"> III. Masking the nifti </span> </br>

In [29]:
# If everything went well, you should have a tmp_dcm2bids folder
# And inside the folder ../subject_tag, you need to select the magnitude you want to load for masking

path_to_magnitude = r"/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/tmp_dcm2bids/sub-chi12/006_I_sorted_dicoms_may29th_2D_7meGRE_no_custom_shim_20260528150928_e1.nii.gz"

In [30]:
outpath_for_sc_msk = os.path.join(transfer_home_directory, "I_niftis_may_28th", "sc_mask.nii.gz")
print(outpath_for_sc_msk)

/Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/sc_mask.nii.gz


In [ ]:
# We need to split along the time dimensions first

In [22]:
!sct_image -i {path_to_magnitude} -split t


--
Spinal Cord Toolbox (git-sr/qc-dev-c8e384aa3d7977d2eb14dc0976e99e4825eab986)

sct_image -i /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/sub-chi12/fmap/sub-chi12_magnitude2.nii.gz -split t
--

Generate output files...
Total runtime; 0.113 seconds.


In [ ]:
# This creates files with _T000x at the end of the name before the .nii.gz extension, we just use the first echo for segmentation

In [ ]:
mag_e1_filename = ".nii.gz"
path_to_mag_e1 = os.path.join(outpath_for_niftis, "tmp_dcm2bids", subject_tag, mag_e1_filename)

In [31]:
!sct_deepseg spinalcord -i {path_to_magnitude} -o {outpath_for_sc_msk} 


--
Spinal Cord Toolbox (git-sr/qc-dev-c8e384aa3d7977d2eb14dc0976e99e4825eab986)

sct_deepseg spinalcord -i /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/tmp_dcm2bids/sub-chi12/006_I_sorted_dicoms_may29th_2D_7meGRE_no_custom_shim_20260528150928_e1.nii.gz -o /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/sc_mask.nii.gz
--

Model 'model_seg_sc_contrast_agnostic_nnunet' is up to date (Source: https://github.com/sct-pipeline/contrast-agnostic-softseg-spinalcord/releases/download/v3.0/model_contrast_agnostic_20250123.zip)
NumExpr defaulting to 10 threads.
perform_everything_on_device=True is only supported for cuda devices! Setting this to False
Running inference on device: cpu
Model loaded successfully.
Creating temporary folder (/var/folders/pc/cz8kpv9d4lzgtnz_gf6t7j1c0000gn/T/sct_2026-05-28_16-19-15_sct_deepseg_5mejbjko)
Copied /Users/mclogar/msc_data/SYNGO_TRANSFER/SYNGO_TRANSFER/I_niftis_may_28th/tmp_dcm2bids/sub-chi12/006_I_sorted_dic

## <span style="color:#AE74D4"> IV. Creating a fieldmap </span> </br>

In [ ]:
path_to_phs = r
path_to_mag = r

path_to_mask = r"/Users/mclogar/msc_data/SYNGO_TRANSFER/phantom_may_25th/complete_sorted_niftis_may_29th/tmp_dcm2bids/sub-h20ball/ball_mask.nii.gz" # outpath_for_sc_msk
unwrapper = "prelude" # better than skimage and similar to ROMEO, but ROMEO not yet implemented
fieldmap_outpath = r"/Users/mclogar/msc_data/SYNGO_TRANSFER/phantom_may_25th/complete_sorted_niftis_may_29th/tmp_dcm2bids/sub-h20ball/st_fm.nii.gz"

In [4]:
!st_prepare_fieldmap {path_to_phs} --mag {path_to_mag} --mask {path_to_mask} -o {fieldmap_outpath}

INFO:shimmingtoolbox.load_nifti:Scaling the selected NIfTI
INFO:shimmingtoolbox.unwrap.unwrap_phase:Unwrapping 1 volume
Traceback (most recent call last):
  File "/Users/mclogar/shimming-toolbox/bin/st_prepare_fieldmap", line 8, in <module>
    sys.exit(prepare_fieldmap_cli())
             ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mclogar/shimming-toolbox/python/lib/python3.11/site-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mclogar/shimming-toolbox/python/lib/python3.11/site-packages/click/core.py", line 1406, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/Users/mclogar/shimming-toolbox/python/lib/python3.11/site-packages/click/core.py", line 1269, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mclogar/shimming-toolbox/python/lib/python3.11/site-packages/click/core.py", line 824, in invoke
    r

For some reason, I can't run this on the jupyter notebook but you can take the code and just use prelude from the terminal

## <span style="color:#AE74D4"> V. Calculating new shimm coefficients </span> </br>

In [10]:
# Lastly, we can use all the created files to create the custom shimm coefficients for the acquisition
fieldmap_outpath = r"/Users/mclogar/msc_data/SYNGO_TRANSFER/phantom_may_25th/complete_sorted_niftis_may_29th/tmp_dcm2bids/sub-h20ball/test_prelude_ball_fm.nii.gz"
shim_coeff_path = r"/Users/mclogar/msc_data/SYNGO_TRANSFER/phantom_may_25th/complete_sorted_niftis_may_29th/tmp_dcm2bids/sub-h20ball/ball_shim_coeffs.txt"


For optimizer method we have: least_squares | pseudo_inverse | quad_prog | bfgs </br>
From Behrouz and Eva's work (caveat: it was for MRS),they saw that LSQ method lead to underperfom. The QuadProAlgorithm performed better but comparable to the Siemens. PI (pseudo inverse) performed better than quad_prog. </br>

use pseudo_inverse first </br>

Ref: https://onlinelibrary.wiley.com/doi/full/10.1002/mrm.30257

In [12]:
!st_b0shim dynamic --fmap {fieldmap_outpath} --target {path_to_mag} --mask {path_to_mask} --scanner-coil-order 0,1 --optimizer-method pseudo_inverse --output-file-format-scanner slicewise-hrd -o {shim_coeff_path}

INFO:shimmingtoolbox.cli.b0shim:The slices to shim are:
[(1,), (3,), (5,), (7,), (9,), (11,), (13,), (15,), (17,), (0,), (2,), (4,), (6,), (8,), (10,), (12,), (14,), (16,)]
INFO:shimmingtoolbox.cli.b0shim:Saving to text file with format: slicewise-hrd
INFO:shimmingtoolbox.cli.b0shim:Coil txt file(s) are here:
/Users/mclogar/msc_data/SYNGO_TRANSFER/phantom_may_25th/complete_sorted_niftis_may_29th/tmp_dcm2bids/sub-h20ball/ball_shim_coeffs.txt/scanner_shim.txt
INFO:shimmingtoolbox.cli.b0shim:Plotting figure(s)
INFO:shimmingtoolbox.cli.b0shim:Plotting currents
